#QUBO Solved Using Pyomo-CPLIX

In [12]:
import numpy as np
from pyomo.environ import *
from scipy.sparse import csr_matrix, find

# ======================================================
# 1. Load QUBO matrix
# ======================================================
Q_dense = np.loadtxt("QUBO_matrix.txt", dtype=float)
n = Q_dense.shape[0]

Q_sparse = csr_matrix(Q_dense)
rows, cols, vals = find(Q_sparse)

print(f"Loaded QUBO: n={n}, nonzeros={len(vals)}, density={len(vals)/(n*n):.4f}")

# ======================================================
# 2. Build Pyomo model (Binary Quadratic Program)
# ======================================================
model = ConcreteModel("QUBO_Model")
model.I = RangeSet(0, n - 1)
model.x = Var(model.I, domain=Binary)

def qubo_objective(model):
    return quicksum(
        vals[k] * model.x[int(rows[k])] * model.x[int(cols[k])]
        for k in range(len(vals))
    )

model.obj = Objective(rule=qubo_objective, sense=minimize)

# ======================================================
# 3. Solve using CPLEX DIRECT (full solve, default settings)
# ======================================================
solver = SolverFactory("cplex_direct")



print("\nSOLVING QUBO WITH CPLEX DIRECT (FULL SOLVE)...\n")

# Solve model
result = solver.solve(model, tee=True)

# ======================================================
# 4. Extract solution
# ======================================================
active_vars = [i for i in model.I if value(model.x[i]) > 0.5]
objective_value = value(model.obj)

# ======================================================
# 5. Print results
# ======================================================
print("\n===== COMPLETE SOLUTION =====")
print(f"Objective Value : {objective_value:.6f}")
print(f"Active Variables (1s) : {active_vars}")
print(f"Number of Active Bits : {len(active_vars)}")

# ======================================================
# 6. Save results to files
# ======================================================
import json

solution_dict = {
    "objective": float(objective_value),
    "active_variables": active_vars,
    "num_active": len(active_vars)
}

# Save as JSON
with open("qubo_solution.json", "w") as f_json:
    json.dump(solution_dict, f_json, indent=2)

# Save as TXT
with open("qubo_solution.txt", "w") as f_txt:
    f_txt.write(f"Objective Value: {objective_value}\n")
    f_txt.write(f"Active Variables (1s): {active_vars}\n")
    f_txt.write(f"Number of Active Bits: {len(active_vars)}\n")

print("\n✅ Solution saved to qubo_solution.json and qubo_solution.txt")


Loaded QUBO: n=675, nonzeros=310263, density=0.6810

SOLVING QUBO WITH CPLEX DIRECT (FULL SOLVE)...

Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (1.21 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 0 rows and 221 columns.
MIP Presolve added 140644 rows and 70322 columns.
Reduced MIP has 140644 rows, 70776 columns, and 281288 nonzeros.
Reduced MIP has 70776 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.05 sec. (114.23 ticks)
Probing time = 0.12 sec. (6.44 ticks)
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 70322 rows and 0 columns.
Reduced MIP has 70322 rows, 70776 columns, and 210966 nonzeros.
Reduced MIP has 70776 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.21 sec. (170.68 ticks)
Classifier predicts products in MIQP should be linearized.
Probing time = 0.08 sec. (5.30 ticks)
MIP emphasis: balance o